In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
path = './data/house_prices/'

In [3]:
def add_to_class(Class):
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

In [4]:
class KaggleHouse(Dataset):
    def __init__(self, train=True):
        super().__init__()
        self.train = train
        self.data = None
        self.raw_data = pd.read_csv(path + ('train.csv' if train else 'test.csv'))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        get_tensor = lambda x: torch.tensor(x.values.astype(float), dtype=torch.float32)
        if self.train == True:
            label = 'SalePrice'
            features = get_tensor(self.data.drop(columns=[label]))
            labels = get_tensor(self.data[label])
            return features[idx], labels[idx]
        else:
            features = get_tensor(self.data)
            return features[idx]


In [5]:
train_dataset = KaggleHouse(train=True)
test_dataset = KaggleHouse(train=False)

In [6]:
train_dataset.raw_data.shape, test_dataset.raw_data.shape

((1460, 81), (1459, 80))

In [7]:
def preprocess():
    # Remove the ID and label columns
    label = 'SalePrice'
    features = pd.concat(
        (train_dataset.raw_data.drop(columns=['Id', label]),
         test_dataset.raw_data.drop(columns=['Id'])))
    
    # Standardize numerical columns
    numeric_features = features.dtypes[features.dtypes!='object'].index
    features[numeric_features] = features[numeric_features].apply(lambda x: (x - x.mean()) / (x.std()))
    
    # Replace NAN numerical features by 0
    features[numeric_features] = features[numeric_features].fillna(0)

    # Replace discrete features by one-hot encoding
    features = pd.get_dummies(features, dummy_na=True)
    
    # Save preprocessed features
    train_dataset.data = features[:train_dataset.raw_data.shape[0]].copy()
    train_dataset.data[label] = train_dataset.raw_data[label]
    test_dataset.data = features[train_dataset.raw_data.shape[0]:].copy()
    print('Preprocessing complete')

In [8]:
preprocess()
print('Train shape:', train_dataset.data.shape)
print('Test shape:', test_dataset.data.shape)

Preprocessing complete
Train shape: (1460, 331)
Test shape: (1459, 330)


In [9]:
display(train_dataset.data.head(2))
display(test_dataset.data.head(2))
print(train_dataset.data.shape, test_dataset.data.shape)

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_WD,SaleType_nan,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SaleCondition_nan,SalePrice
0,0.067320,-0.184443,-0.217841,0.646073,-0.507197,1.046078,0.896679,0.523038,0.580708,-0.29303,...,True,False,False,False,False,False,True,False,False,208500
1,-0.873466,0.458096,-0.072032,-0.063174,2.187904,0.154737,-0.395536,-0.569893,1.177709,-0.29303,...,True,False,False,False,False,False,True,False,False,181500


,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_Oth,SaleType_WD,SaleType_nan,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SaleCondition_nan
0,-0.873466,0.458096,0.184340,-0.772420,0.39117,-0.340452,-1.113434,-0.569893,0.058332,0.558006,...,False,True,False,False,False,False,False,True,False,False
1,-0.873466,0.500932,0.519702,-0.063174,0.39117,-0.439490,-1.257014,0.032335,1.056991,-0.293030,...,False,True,False,False,False,False,False,True,False,False


(1460, 331) (1459, 330)


In [10]:
features = train_dataset.data.drop(columns=['SalePrice'])
label = train_dataset.data[['SalePrice']]
display(features.head(2))
display(label.head(2))

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_Oth,SaleType_WD,SaleType_nan,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SaleCondition_nan
0,0.067320,-0.184443,-0.217841,0.646073,-0.507197,1.046078,0.896679,0.523038,0.580708,-0.29303,...,False,True,False,False,False,False,False,True,False,False
1,-0.873466,0.458096,-0.072032,-0.063174,2.187904,0.154737,-0.395536,-0.569893,1.177709,-0.29303,...,False,True,False,False,False,False,False,True,False,False


,SalePrice
0,208500
1,181500


In [11]:
def convert_to_tensor(x):
    tensor = torch.tensor(x.values.astype(float), dtype=torch.float32)
    return tensor

In [12]:
features = convert_to_tensor(features)
label = convert_to_tensor(label)

In [13]:
# Loss function
def rmsle_loss(y_pred, y_true):
    loss = torch.sqrt(F.mse_loss(torch.log1p(y_pred), torch.log1p(y_true)))
    return loss

In [14]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.LazyLinear(1)
        self.net.weight.data.normal_(0, 0.01)
        self.net.bias.data.fill_(0)

    def forward(self, X):
        return self.net(X)

In [15]:
folds = 5
skf = KFold(n_splits=folds, shuffle=True, random_state=42)

In [16]:
# Metrics
criterion = rmsle_loss
batch_size = 256

In [ ]:
epochs = 10
Loss = []
models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(features, label)):
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size)
    val_loader = DataLoader(val_subset, batch_size=batch_size)
    model = LinearRegression()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    print(f'FOLD: {fold+1}/{folds}')
    L = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        for i, (features, labels) in enumerate(train_loader):
            optimizer.zero_grad()
            outputs = model(features).view(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        L.append(epoch_loss / len(train_loader))
        
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for features, labels in val_loader:
                outputs = model(features).view(-1)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        print(f'Epoch: {epoch+1}/{epochs}. Train loss: {L[-1]:.4f}%. Val loss: {val_loss:.4f}%')

    models.append(model)
    Loss.append(L)


FOLD: 1/5


In [ ]:
Loss[0]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, ax in enumerate(axes.flat):
    ax.plot(range(1, len(Loss[i]) + 1), Loss[i], label=f"Fold {i+1}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"Fold {i+1}")
    ax.legend()

plt.tight_layout()
plt.show()

In [17]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [18]:
epochs = 10
Loss = []
model = LinearRegression()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(epochs):
    epoch_loss = 0
    for i, (features, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(features).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    Loss.append(epoch_loss / len(train_loader))
    print(f'Epoch: {epoch+1}/{epochs}. Train loss: {Loss[-1]:.4f}%.')

Epoch: 1/10. Train loss: 10.6570%.
Epoch: 2/10. Train loss: 9.9729%.
Epoch: 3/10. Train loss: 9.7451%.
Epoch: 4/10. Train loss: 9.5920%.
Epoch: 5/10. Train loss: 9.4762%.
Epoch: 6/10. Train loss: 9.3787%.
Epoch: 7/10. Train loss: 9.2995%.
Epoch: 8/10. Train loss: 9.2312%.
Epoch: 9/10. Train loss: 9.1699%.
Epoch: 10/10. Train loss: 9.1191%.


In [52]:
def predict(model, test_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for features in test_loader:
            outputs = model(features).view(-1)
            predictions.extend(outputs.numpy().flatten())
    return predictions

In [53]:
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [54]:
preds = predict(model, test_loader)

In [55]:
print(preds)

[18.315742, 19.519043, 18.835018, 18.875519, 18.410013, 17.252605, 18.838932, 17.784733, 18.313364, 17.740738, 18.898443, 17.917337, 17.111746, 17.46109, 17.364344, 20.01743, 18.539684, 18.459042, 17.952984, 16.60045, 18.637459, 19.515118, 17.778976, 16.135452, 18.577723, 19.778107, 17.923698, 18.663687, 16.903347, 17.293264, 19.475517, 18.928528, 18.959148, 19.944834, 19.002785, 14.513481, 16.953594, 14.527914, 16.661003, 19.172867, 18.410664, 18.5584, 18.930054, 17.218782, 17.78271, 17.873432, 16.726686, 18.8206, 17.57336, 19.801186, 18.737854, 19.311224, 16.841604, 14.411959, 18.888811, 18.88845, 19.450672, 19.25085, 18.224384, 18.681787, 18.238699, 16.439655, 18.135384, 18.977081, 16.847893, 19.852947, 18.770786, 16.54455, 19.12991, 18.794464, 18.628872, 18.910347, 17.171158, 19.244781, 15.607099, 18.718525, 17.712433, 19.326153, 17.457348, 18.45136, 18.357225, 17.282888, 18.473099, 19.547554, 15.513559, 18.03783, 17.835838, 18.584408, 19.53867, 18.533842, 18.863634, 18.12042, 17.4